In [1]:
#  << 데이터 가용성 확보를 위한 경로 리스트 구축 및 데이터셋 규격화 >>
# 소규모 데이터셋 환경에서의 데이터 효율적 AI 모델링 및 학습 속도 최적화를 위해 클래스별 500개 샘플 추출
import os
import random

# 메모리 점유율 최소화를 위해 전체 픽셀 데이터 대신 파일 경로 리스트만 사전 구축
# 학습 시점에 텐서 변환 수행할 예정
def get_sampled_files(directory, sample_size=500):
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            all_files.append(os.path.join(root, file))
    
    # [학습할 데이터량] 
    # 랜덤하게 500개만 섞어서 추출
    # 만약 원래 개수가 500개 이하라면 전체 반환
    if len(all_files) > sample_size:
        sampled_files = random.sample(all_files, sample_size)
    else:
        sampled_files = all_files
        
    return sampled_files

FIGHTER_DIR = '/kaggle/input/datasets/jrmymimran/fighterjets'
DRONE_DIR = '/kaggle/input/datasets/dasmehdixtr/drone-dataset-uav'
ROCKET_DIR = '/kaggle/input/datasets/eneskosar19/rocket-dataset-for-image-detection-labelled'

# 프로토타입 단계에서의 PoC(Proof of Concept) 가속화를 위해 클래스별 균등 서브셋 500개로 구축
fighter_500 = get_sampled_files(FIGHTER_DIR, 500)
drone_500 = get_sampled_files(DRONE_DIR, 500)
rocket_500 = get_sampled_files(ROCKET_DIR, 500)

print(f"추출 완료: 전투기({len(fighter_500)}), 드론({len(drone_500)}), 로켓({len(rocket_500)})")

추출 완료: 전투기(500), 드론(500), 로켓(500)


In [ ]:
# << 원본 이미지의 기하학적 형상 보존(Zero-Padding) 및 데이터 무결성 검증(Fail-Safe)이 통합된 딥러닝 전처리 파이프라인 구축>>
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os

# 클래스 [1] : 원본 이미지 형상을 보존하며 추론 엔진용 텐서로 변환하는 데이터 공급 클래스
class AeroObjectDataset(Dataset):
    def __init__(self, file_paths, labels, img_size=224, is_train=False):
        self.file_paths = file_paths
        self.labels = labels
        self.img_size = img_size
        self.is_train = is_train
        
        # [데이터 증강 및 텐서 변환 파이프라인]
        if self.is_train:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                # 50% 확률로 수평 반전
                transforms.RandomRotation(15), 
                # 요격 각도 변화 모사 -> 각도를 너무 많이 잡으면 원본과 동 떨어진 오버피팅 우려해 15도로 초기화 
                transforms.ColorJitter(brightness=0.2, contrast=0.2), 
                # 밝기/대비 변화 모사 -> 1.0으로 잡아버리면 원본이미지 훼손이 있어 학습 퀄리티 저해할 것 대비 0.2로 초기화
                transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 3)), 
                # kernel_size(눈의 시야) -> 작은 시야로 세밀하게 볼 5부터 큰 시야로 넓게도 보도록 9까지로 잡음
                # sigma(번짐의 강도) -> 블러가 거의 없는 0.1에서 아주 강력한 번짐의 3까지로 잡음.
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                # Tesnor : 3차원(층, 줄, 칸) 공간 주소에 배정된 RRB 색상값들을 엔진의 표준 규격에 맞춰 교정
                # Resnet-18의 평균 값 -> [0.485, 0.456, 0.406]
                # Resnet-18의 평균 표준편차 -> [0.229, 0.224, 0.225]
            ])
        else:
            # 검증/테스트 시에는 증강 없이 정규화만 수행
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    # [ 종횡비 유지를 위한 중앙 정렬 제로 패딩 ]
    def _zero_padding(self, image):
        
        w, h = image.size
        max_dim = max(w, h)
        
        # 검은색(0,0,0) 정사각형 배경을 생성하고 원본 이미지를 중앙에 배치하여 종횡비 보존
        new_image = Image.new('RGB', (max_dim, max_dim), (0, 0, 0))
        
        # 원본 이미지를 캔버스 중앙에 배치
        paste_pos = ((max_dim - w) // 2, (max_dim - h) // 2)
        new_image.paste(image, paste_pos)

        # 최종 모델 입력 사이즈(224x224)로 리사이징
        # 이미지 작으면 확대 / 이미지 크면 축소 -> 기존에도 정사각형 형태라 원본에 훼손이 없음
        return new_image.resize((self.img_size, self.img_size))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        
        # Fail-safe: 비정상 파일 유입 시 예외 처리 및 차단
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"파일 손상으로 무작위 대체: {os.path.basename(img_path)}")
            import random
            random_idx = random.randint(0, len(self.file_paths) - 1)
            return self.__getitem__(random_idx) # 다른 정상 샘플로 대체하여 반환
            
        padded_image = self._zero_padding(image) # 제로 패딩 적용 
        tensor_image = self.transform(padded_image) # 텐서 변환 및 데이터 증강
        
        return tensor_image, torch.tensor(self.labels[idx], dtype=torch.long)


# 클래스 [2] : 데이터 무결성을 검증하고 손상된 샘플을 데이터셋에서 영구 제외하는 모듈
class FailSafeValidator:
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels
        self.clean_paths = []
        self.clean_labels = []
        
    # 전체 리스트를 순회하며 실제 열리는 이미지인지 검증 (학습 시작 전 동작)
    def filter_bad_images(self):
        print(f"[*] 데이터 무결성 검사 시작 (대상: {len(self.file_paths)}개)...")
        
        for path, label in zip(self.file_paths, self.labels):
            try:
                # 1. 파일 존재 여부 확인
                if not os.path.exists(path):
                    raise FileNotFoundError
                
                # 2. 실제로 이미지가 열리고 RGB 변환이 가능한지 확인
                with Image.open(path) as img:
                    img.verify() # 파일 구조 1차 검증
                with Image.open(path) as img:
                    img.load() # 실제 픽셀 데이터 손상 여부 2차 검증
                
                # 검증 통과 시 '깨끗한 리스트'에 추가
                self.clean_paths.append(path)
                self.clean_labels.append(label)
                
            except Exception as e:
                # 국방 시스템 로그 기록 모사
                print(f"무결성 검증 실패 - 제외 처리됨: {os.path.basename(path)}")
                continue

        removed_count = len(self.file_paths) - len(self.clean_paths)
        print(f"검사 완료 : {removed_count}개의 불량 샘플 제거됨.")
        return self.clean_paths, self.clean_labels